# 01 — Model Anatomy: Tokenization & Architecture

**Goal:** Understand what the pieces of an LLM are before we learn how they work.

By end of this notebook you'll know:
- What a token is and how text becomes numbers
- What the model's architecture looks like (16 layers stacked)
- What each layer contains (attention + MLP)
- That the model can actually generate text

---
## Part 1: Load the Model

In [ ]:
import sys
sys.path.insert(0, '../src')

from inspector.model_loader import load_model, get_model_info, generate_text

model, tokenizer = load_model()

---
## Part 2: Model Configuration

Let's see what this model is made of — how many layers, attention heads, etc.

In [ ]:
# Print the key configuration values
info = get_model_info(model)

In [ ]:
# Print the FULL architecture — every layer, every weight matrix
# This is long but important to see once.
# Notice the repeating pattern: each layer has self_attn + mlp
print(model)

### What you just saw:

The model is a **stack of identical layers**. Each layer has:

1. **Self-Attention** (`self_attn`) — this is where tokens "look at" other tokens
   - `q_proj`, `k_proj`, `v_proj` — the Query, Key, Value projections
   - `o_proj` — output projection that combines the attention results

2. **MLP** (Feed-Forward Network) — this is where the model "thinks"
   - `gate_proj`, `up_proj` — expand the representation to a larger space
   - `down_proj` — compress back down

3. **Layer Norms** (`input_layernorm`, `post_attention_layernorm`) — keep numbers stable

Plus two special layers:
- **`embed_tokens`** at the start — converts token IDs → vectors
- **`lm_head`** at the end — converts vectors → next-word predictions

---
## Part 3: Tokenization — How Text Becomes Numbers

The model can't read text. It works with numbers. **Tokenization** is the process of
converting text → numbers (token IDs) that the model understands.

In [ ]:
# Basic encoding: text → token IDs
text = "Why is the sun yellow"
token_ids = tokenizer.encode(text)
print(f"Text:      '{text}'")
print(f"Token IDs: {token_ids}")
print(f"Number of tokens: {len(token_ids)}")

In [ ]:
# Decoding: token IDs → text
decoded = tokenizer.decode(token_ids)
print(f"Decoded back: '{decoded}'")

In [ ]:
# See each token individually — this is the key insight!
# Notice: each token ID maps to a piece of text (sometimes a word, sometimes part of a word)
print("Token-by-token breakdown:")
print("-" * 40)
for i, tid in enumerate(token_ids):
    token_text = tokenizer.decode([tid])
    print(f"  Position {i}: ID={tid:>6}  →  '{token_text}'")

### Subword Tokenization — Why Words Get Split

The model doesn't know whole words. It knows **subwords** — common pieces of words.
This lets it handle any word, even ones it's never seen before.

- Common words stay whole: "the", "is", "cat"
- Uncommon words get split: "understanding" → "under" + "standing"
- Very rare words get split more: "pneumonia" → "pne" + "um" + "onia"

Let's see this in action:

In [ ]:
# Examples of subword tokenization
examples = [
    "hello",
    "understanding",
    "transformers",
    "pneumonoultramicroscopicsilicovolcanoconiosis",
    "The quick brown fox jumps over the lazy dog",
    "def hello_world():\n    print('Hello!')",
    "https://www.example.com/path?query=value",
]

for text in examples:
    tokens = tokenizer.encode(text)
    pieces = [tokenizer.decode([t]) for t in tokens]
    print(f"\n'{text}'")
    print(f"  → {len(tokens)} tokens: {pieces}")

In [ ]:
# How big is the vocabulary?
vocab = tokenizer.get_vocab()
print(f"Vocabulary size: {len(vocab):,} tokens")

# Show some example tokens from the vocabulary
print(f"\nFirst 20 tokens (by ID):")
# Sort by token ID and show first 20
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
for token_text, token_id in sorted_vocab[:20]:
    print(f"  ID={token_id:>5}: '{token_text}'")

In [ ]:
# Special tokens — tokens with special meaning
print("Special tokens:")
print(f"  BOS (Beginning of Sequence): '{tokenizer.bos_token}' (ID: {tokenizer.bos_token_id})")
print(f"  EOS (End of Sequence):       '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
print(f"  PAD (Padding):               '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

# Show that encoding adds BOS automatically
text = "Hello"
with_special = tokenizer.encode(text, add_special_tokens=True)
without_special = tokenizer.encode(text, add_special_tokens=False)
print(f"\n'Hello' with special tokens:    {with_special}")
print(f"'Hello' without special tokens: {without_special}")

---
## Part 4: From Tokens to Embeddings

Token IDs are just numbers (like dictionary indices). The model needs **vectors** — 
lists of numbers that capture meaning.

The **embedding table** is a giant lookup table:
- 128,256 rows (one per token in the vocabulary)
- 2,048 columns (the "hidden size" — how many numbers describe each token)

When the model sees token ID 5678, it looks up row 5678 in this table and gets 
a vector of 2,048 numbers. That vector IS the model's understanding of that token.

In [ ]:
import torch

# The embedding table
embed_table = model.model.embed_tokens.weight
print(f"Embedding table shape: {embed_table.shape}")
print(f"  → {embed_table.shape[0]:,} tokens, each represented by {embed_table.shape[1]} numbers")
print(f"  → Total parameters in embeddings: {embed_table.numel():,}")
print(f"  → Memory: {embed_table.numel() * 2 / 1024 / 1024:.1f} MB (FP16)")

# Look up the embedding for a specific word
word = "Paris"
token_id = tokenizer.encode(word, add_special_tokens=False)[0]
embedding = embed_table[token_id]
print(f"\nEmbedding for '{word}' (ID={token_id}):")
print(f"  Shape: {embedding.shape}")
print(f"  First 10 values: {embedding[:10].tolist()}")
print(f"  Min: {embedding.min():.4f}, Max: {embedding.max():.4f}, Mean: {embedding.float().mean():.4f}")

---
## Part 5: The Full Pipeline — Text In, Text Out

Here's how the model processes text:

```
"The capital of France is" 
    → tokenize → [128000, 791, 6864, 315, 9822, 374]
    → embed    → 6 vectors of size 2048
    → layer 0  → 6 vectors (slightly transformed)
    → layer 1  → 6 vectors (more transformed)
    → ...      → ...
    → layer 15 → 6 vectors (fully processed)
    → lm_head  → probability over 128,256 possible next tokens
    → argmax   → token ID with highest probability
    → decode   → "Paris"
```

Let's watch this happen:

In [ ]:
# Run the model and capture everything
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

# What did we get back?
print("Output keys:", [k for k in outputs.keys()])
print(f"\nLogits shape: {outputs.logits.shape}")
print(f"  → batch_size=1, sequence_length={outputs.logits.shape[1]}, vocab_size={outputs.logits.shape[2]}")
print(f"\nHidden states: {len(outputs.hidden_states)} tensors (embedding + 16 layers)")
print(f"  Each shape: {outputs.hidden_states[0].shape}")
print(f"\nAttentions: {len(outputs.attentions)} tensors (one per layer)")
print(f"  Each shape: {outputs.attentions[0].shape}")
print(f"  → batch=1, heads={outputs.attentions[0].shape[1]}, seq={outputs.attentions[0].shape[2]}, seq={outputs.attentions[0].shape[3]}")

In [ ]:
# What does the model predict as the next word?
# Take the logits for the LAST token position (that's the prediction for what comes next)
last_token_logits = outputs.logits[0, -1, :]  # shape: [vocab_size]

# Get top 10 predictions
top_probs = torch.softmax(last_token_logits.float(), dim=-1)
top_values, top_indices = torch.topk(top_probs, k=10)

print(f"Prompt: '{prompt}'")
print(f"\nTop 10 next-word predictions:")
print("-" * 45)
for i, (prob, idx) in enumerate(zip(top_values, top_indices)):
    token_text = tokenizer.decode([idx])
    bar = '█' * int(prob * 50)
    print(f"  {i+1}. '{token_text}' ({prob:.1%}) {bar}")

---
## Part 6: Test Generation

Let's run the model on several prompts to confirm it works and see what it produces.

In [ ]:
import json

# Load test prompts
with open('../data/test_prompts.json') as f:
    test_prompts = json.load(f)

# Run a few prompts from each category
for category, prompts in test_prompts.items():
    print(f"\n{'='*60}")
    print(f"Category: {category.upper()}")
    print(f"{'='*60}")
    for prompt in prompts[:2]:  # just first 2 per category to save time
        result = generate_text(model, tokenizer, prompt, max_new_tokens=30)
        print(f"\n  Prompt: '{prompt}'")
        # Show only the generated part (after the prompt)
        generated = result[len(prompt):]
        print(f"  Generated: '{generated.strip()}'")

---
## Part 7: Parameter Count Breakdown

Where do all the parameters live? Let's count them per component.

In [ ]:
# Count parameters by component
def count_params(module):
    return sum(p.numel() for p in module.parameters())

total = count_params(model)
embed = count_params(model.model.embed_tokens)
lm_head = count_params(model.lm_head)
norm = count_params(model.model.norm)

print(f"Total parameters: {total:,} ({total * 2 / 1024**3:.2f} GB in FP16)")
print(f"\nBreakdown:")
print(f"  Embeddings:  {embed:>12,} ({embed/total:.1%})")
print(f"  LM Head:     {lm_head:>12,} ({lm_head/total:.1%})")
print(f"  Final Norm:  {norm:>12,} ({norm/total:.1%})")

# Per-layer breakdown
print(f"\n  Per-layer breakdown (Layer 0 as example):")
layer0 = model.model.layers[0]
attn = count_params(layer0.self_attn)
mlp = count_params(layer0.mlp)
norms = count_params(layer0.input_layernorm) + count_params(layer0.post_attention_layernorm)
layer_total = count_params(layer0)
print(f"    Attention:   {attn:>10,} ({attn/layer_total:.1%} of layer)")
print(f"    MLP:         {mlp:>10,} ({mlp/layer_total:.1%} of layer)")
print(f"    Layer Norms: {norms:>10,} ({norms/layer_total:.1%} of layer)")
print(f"    Layer Total: {layer_total:>10,} ({layer_total/total:.1%} of model)")
print(f"    × 16 layers: {layer_total * 16:>10,} ({layer_total * 16/total:.1%} of model)")

---
## Summary

### What we learned:

1. **Tokenization** converts text into numbers. The model uses ~128K subword tokens.
   Common words stay whole, rare words get split into pieces.

2. **The model architecture** is a stack of 16 identical layers, each containing:
   - Self-attention (Q, K, V projections — we'll learn what these do on Day 2)
   - MLP (feed-forward network — expands then compresses the representation)
   - Layer norms (keep the numbers from exploding)

3. **The pipeline**: text → tokens → embeddings → 16 layers → final prediction

4. **The embedding table** maps each of 128K tokens to a vector of 2,048 numbers.
   These vectors capture the "meaning" of each token.

5. **The model works** — it can complete sentences, do math (sometimes), and write code.

### What's next (Day 2):
- Build the **logit lens** to see what each layer predicts
- Understand **attention** — the Q, K, V mechanism
- See the model "figure out" answers layer by layer